In [1]:
# =====================================
# IMPORTS
# =====================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# =====================================
# LOAD PREPROCESSED DATA
# =====================================

model_data = pd.read_pickle("../data/preprocessed_model_data.pkl")

print("Loaded preprocessed data:")
print(model_data.shape)
print(model_data.columns.tolist())


Loaded preprocessed data:
(45924, 24)
['Confirmation Year', 'Handler Region', 'Product Group', 'Product Type', 'Product Type Code', 'Industry Name', 'Industry Code', 'Service Line Code', 'Service Line Name', 'Service Detail', 'Service Program', 'Service Catalog Category', 'Service Catalog Item Number', 'Service Catalog Segment', 'Service Catalog Sub Category', 'CCN', 'Ship to Customer Region', 'Has Test Task Flag', 'Ship to Account Number', 'Flex Standards', 'Standard Count', 'Flex Project Count', 'Test Count', 'Log_Eng_Hours']


In [3]:
# =====================================
# CREATE X AND y
# =====================================

X = model_data.drop(columns=["Log_Eng_Hours"])
y = model_data["Log_Eng_Hours"]

cat_cols = X.select_dtypes(include=["object", "str", "category"]).columns

X_encoded = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

# Clean column names for XGBoost
X_encoded.columns = (
    X_encoded.columns
    .astype(str)
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

print("X shape:", X_encoded.shape)
print("y shape:", y.shape)

X shape: (45924, 2616)
y shape: (45924,)


In [4]:
# =====================================
# TRAIN / TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (36739, 2616)
X_test: (9185, 2616)
y_train: (36739,)
y_test: (9185,)


In [5]:
#Evaluation Function

def evaluate_model(model, X_test, y_test):
    y_pred_log = model.predict(X_test)

    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_test, y_pred_log)

    return mae, rmse, r2


In [6]:
# =====================================
# TRAIN MODELS
# =====================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001, max_iter=10000),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )
}

results_list = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    mae, rmse, r2 = evaluate_model(model, X_test, y_test)

    results_list.append({
        "Model": name,
        "MAE_hours": mae,
        "RMSE_hours": rmse,
        "R2_log_scale": r2
    })

results = pd.DataFrame(results_list).sort_values("RMSE_hours")

results


Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training Random Forest...
Training XGBoost...


,Model,MAE_hours,RMSE_hours,R2_log_scale
3,Random Forest,5.988109,15.599409,0.555829
4,XGBoost,6.441747,16.666157,0.504787
0,Linear Regression,6.843460,16.982738,0.430132
1,Ridge Regression,6.817558,17.094699,0.440819
2,Lasso Regression,7.291750,18.292007,0.377858


In [7]:
# ============================================================
# DEVIATION BUCKET SUMMARY TABLES
# 100% Data, 70% Training Data, 30% Test Data
# ============================================================

import pandas as pd
import numpy as np

# Use best model
best_model = models["Random Forest"]

# Helper function to create bucket summary
def create_deviation_summary(X_data, y_data, label):
    # Predict log hours
    y_pred_log = best_model.predict(X_data)

    # Convert log hours back to real hours
    actual_hours = np.expm1(y_data)
    predicted_hours = np.expm1(y_pred_log)

    # Create results table
    temp = pd.DataFrame({
        "Actual Human": actual_hours,
        "Model Estimate": predicted_hours
    })

    # Difference between actual and predicted
    temp["Deviation"] = abs(temp["Actual Human"] - temp["Model Estimate"])

    # Buckets
    temp["Deviation Buckets"] = pd.cut(
        temp["Deviation"],
        bins=[-0.001, 1, 2, 3, 7, np.inf],
        labels=["<1 hour", "1-2 hours", "2-3 hours", "3-7 hours", ">7 hours"]
    )

    # Summary
    summary = temp.groupby("Deviation Buckets").agg(
        **{
            "# Projects": ("Deviation", "count"),
            "Actual Human": ("Actual Human", "mean"),
            "Model Estimate Median": ("Model Estimate", "median")
        }
    ).reset_index()

    # Percent of projects
    summary["% of Projects"] = (
        summary["# Projects"] / summary["# Projects"].sum() * 100
    )

    # Reorder columns
    summary = summary[
        ["Deviation Buckets", "# Projects", "% of Projects", "Actual Human", "Model Estimate Median"]
    ]

    # Round
    summary["% of Projects"] = summary["% of Projects"].round(0).astype(int).astype(str) + "%"
    summary["Actual Human"] = summary["Actual Human"].round(2)
    summary["Model Estimate Median"] = summary["Model Estimate Median"].round(2)

    print("\n" + "="*60)
    print(label)
    print("="*60)
    display(summary)

    return summary


# 100% of data
summary_100 = create_deviation_summary(
    X_encoded,
    y,
    "100% of Data"
)

# 70% training data
summary_train = create_deviation_summary(
    X_train,
    y_train,
    "70% Used in Training"
)

# 30% test data
summary_test = create_deviation_summary(
    X_test,
    y_test,
    "30% Not Used in Training"
)



100% of Data


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,19238,42%,6.04,4.40
1,1-2 hours,8947,19%,8.64,6.85
2,2-3 hours,4971,11%,11.07,9.05
3,3-7 hours,7397,16%,15.43,11.87
4,>7 hours,5371,12%,40.15,19.74



70% Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,16649,45%,6.12,4.47
1,1-2 hours,7375,20%,9.02,7.14
2,2-3 hours,3935,11%,11.84,9.70
3,3-7 hours,5358,15%,17.15,13.18
4,>7 hours,3422,9%,46.01,23.76



30% Not Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,2589,28%,5.55,3.94
1,1-2 hours,1572,17%,6.83,5.50
2,2-3 hours,1036,11%,8.18,6.78
3,3-7 hours,2039,22%,10.93,9.14
4,>7 hours,1949,21%,29.86,15.21


In [8]:
# =====================================
# FEATURE IMPORTANCE FOR BEST MODEL
# =====================================

rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)


,Feature,Importance
3,Test Count,0.136038
0,Ship to Account Number,0.127663
1849,Service Catalog Segment_TST,0.069041
2027,Service Catalog Sub Category_Global Market Ser...,0.051718
2415,Has Test Task Flag_Yes,0.028167
1307,Service Program_New Construction,0.025030
2206,CCN_AAAE,0.014778
1,Standard Count,0.013667
5,Confirmation Year_2026,0.013322
6,Handler Region_Americas,0.010644
